In [ ]:
MODEL_NAMES = {
    "deepseek_7b": "DeepSeek", "mistral_7b": "Mistral", "phi35_mini": "Phi-3.5",
    "gemma3_4b": "Gemma-3", "llama31_8b": "Llama-3.1", "qwen2_7b": "Qwen2",
    "olmo2_7b": "OLMo-2",
}

CONDITION_NAMES = {
    "asd": "Autism",
    "dyslexia": "Dyslexia",
    "down syndrome": "Down Syndrome",
    "Learning Difficulties": "Learning Difficulties",
    "Challenging Behavior": "Challenging Behavior",
    "control": "Control"
    
}

WARMTH = ["tolerant","warm","good-natured","sincere"]
COMPETENCE = ["competent","confident","independent","intelligent"]
ADJECTIVES = WARMTH + COMPETENCE
MIDPOINT = 3.5
MIN_N = 10

SCHELL = ["control","dyslexia","asd","down_syndrome"]
KRISCHLER = ["control","challenging_behavior","learning_difficulties"]
BLOCKS = {"Schell": SCHELL, "Krischler": KRISCHLER}

INPUT_GLOB = "output/simplified/experiment_results_*.csv"

In [ ]:
import glob, json, re
import pandas as pd
import numpy as np
import pingouin as pg
from scipy import stats
from statsmodels.stats.multitest import multipletests

pd.set_option("display.width", 180); pd.set_option("display.max_columns", None)

In [ ]:
def _col(df, *names):
    for n in names:
        if n in df.columns: return n
    raise KeyError(f"nessuna di {names} in {df.columns.tolist()}")

def extract_json_object(text):
    if not isinstance(text, str): return None
    start = text.find("{")
    if start == -1: return None
    depth, in_str, esc = 0, False, False
    for i in range(start, len(text)):
        ch = text[i]
        if in_str:
            if esc: esc = False
            elif ch == "\\": esc = True
            elif ch == '"': in_str = False
            continue
        if ch == '"': in_str = True
        elif ch == "{": depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0: return text[start:i+1]
    return None

def robust_parse(text):
    cand = extract_json_object(text)
    if cand is None: return None
    try: return json.loads(cand)
    except json.JSONDecodeError:
        rep = re.sub(r'(:\s*\d+),(\d+)(?=\s*[,}])', r'\1.\2', cand)
        try: return json.loads(rep)
        except json.JSONDecodeError: return None

def extract_adjectives(raw):
    data = robust_parse(raw)
    if data is None or "error_parsing" in data:
        return pd.Series([np.nan]*8, index=ADJECTIVES)
    return pd.Series([pd.to_numeric(data.get(a, np.nan), errors="coerce") for a in ADJECTIVES],
                     index=ADJECTIVES)

def load_data():
    files = glob.glob(INPUT_GLOB)
    if not files:
        raise FileNotFoundError(f"nessun file per {INPUT_GLOB}")
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    scm_col = _col(df, "scm_raw", "scm_json")
    df[ADJECTIVES] = df[scm_col].apply(extract_adjectives)
    return df

def diagnostics(df):
    print("DIAGNOSTICS - failed parsing ratio")
    fail = df.groupby(["model_id","condition"])["competent"].apply(lambda s: s.isna().mean()).unstack()
    print(fail.round(3))
    if (fail > 0.1).any().any():
        print("WARNING: high failed parsing rate.")
    
def absolute_stereotyping(df):
    rows = []
    for block, conds in BLOCKS.items():
        sub_b = df[df["condition"].isin(conds)]
        for model in sub_b["model_id"].unique():
            dm = sub_b[sub_b["model_id"] == model]
            for cond in conds:
                dc = dm[dm["condition"] == cond]
                for adj in ADJECTIVES:
                    v = dc[adj].dropna()
                    if len(v) < MIN_N or v.nunique() < 2:
                        continue
                    res = pg.ttest(v, MIDPOINT)
                    delta_val = v.mean() - MIDPOINT
                    raw_d = res[_col(res,"cohen_d","cohen-d")].values[0]
                    signed_d = raw_d if delta_val >= 0 else -raw_d
                    
                    rows.append({
                        "block": block,
                        "model_id": model,
                        "condition": cond,
                        "adjective": adj,
                        "mean": v.mean(),
                        "n": len(v),
                        "t": res["T"].values[0],
                        "p_raw": res[_col(res,"p_val","p-val")].values[0],
                        "cohen_d": signed_d,  
                        "direction": "above" if delta_val > 0 else "below"
                    })
    out = pd.DataFrame(rows)
    if len(out):
        rej,pf,_,_ = multipletests(out["p_raw"],alpha=0.05,method="fdr_bh")
        out["p_fdr"]=pf; out["sig_fdr"]=rej
        out["relevant"]= out["sig_fdr"] & (out["cohen_d"].abs()>=0.2)
    return out

def relative_discrimination(df):
    rows = []
    for block, conds in BLOCKS.items():
        sub_b = df[df["condition"].isin(conds)]
        for model in sub_b["model_id"].unique():
            dm = sub_b[sub_b["model_id"] == model]
            ctrl = dm[dm["condition"] == "control"]
            for cond in [c for c in conds if c != "control"]:
                dc = dm[dm["condition"] == cond]
                for adj in ADJECTIVES:
                    a = ctrl[adj].dropna(); b = dc[adj].dropna()
                    if len(a) < MIN_N or len(b) < MIN_N or (a.nunique()<2 or b.nunique()<2):
                        continue
                    res = pg.ttest(b, a, correction=True)  
                    delta_val = b.mean() - a.mean()
                    signed_g = pg.compute_effsize(b, a, eftype="hedges") 
                    rows.append({
                        "block": block,
                        "model_id": model,
                        "condition_vs_control": cond,
                        "adjective": adj,
                        "mean_control": a.mean(),
                        "mean_condition": b.mean(),
                        "delta": delta_val,
                        "hedges_g": signed_g,  
                        "p_raw": res[_col(res,"p_val","p-val")].values[0]
                    })
    out = pd.DataFrame(rows)
    if len(out):
        rej,pf,_,_ = multipletests(out["p_raw"],alpha=0.05,method="fdr_bh")
        out["p_fdr"]=pf; out["sig_fdr"]=rej
        out["relevant"]= out["sig_fdr"] & (out["hedges_g"].abs()>=0.2)
    return out

In [ ]:
df = load_data()
diagnostics(df)

In [ ]:
abs_res = absolute_stereotyping(df)
rel_res = relative_discrimination(df)
abs_res.to_csv("rq1_absolute.csv", index=False)
rel_res.to_csv("rq1_relative.csv", index=False)

In [ ]:
import glob, json, re
import pandas as pd
import numpy as np
import pingouin as pg
from statsmodels.stats.multitest import multipletests

pd.set_option("display.width", 180); pd.set_option("display.max_columns", None)


BLOCKS = {
    "Schell":    ["asd", "dyslexia", "down_syndrome"],
    "Krischler": ["challenging_behavior", "learning_difficulties"],
}


def _col(df, *names):
    for n in names:
        if n in df.columns: return n
    raise KeyError(names)

def extract_json_object(text):
    if not isinstance(text, str): return None
    s = text.find("{")
    if s == -1: return None
    depth, ins, esc = 0, False, False
    for i in range(s, len(text)):
        c = text[i]
        if ins:
            if esc: esc = False
            elif c == "\\": esc = True
            elif c == '"': ins = False
            continue
        if c == '"': ins = True
        elif c == "{": depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0: return text[s:i+1]
    return None

def robust_parse(text):
    cand = extract_json_object(text)
    if cand is None: return None
    try: return json.loads(cand)
    except json.JSONDecodeError:
        rep = re.sub(r'(:\s*\d+),(\d+)(?=\s*[,}])', r'\1.\2', cand)
        try: return json.loads(rep)
        except json.JSONDecodeError: return None

def extract_adjectives(raw):
    d = robust_parse(raw)
    if d is None or "error_parsing" in d:
        return pd.Series([np.nan]*8, index=ADJECTIVES)
    return pd.Series([pd.to_numeric(d.get(a, np.nan), errors="coerce") for a in ADJECTIVES], index=ADJECTIVES)

def load_data():
    files = glob.glob(INPUT_GLOB)
    if not files: raise FileNotFoundError(INPUT_GLOB)
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    raw = "scm_raw" if "scm_raw" in df.columns else "scm_json"
    df[ADJECTIVES] = df[raw].apply(extract_adjectives)
    for a in ADJECTIVES:
        df[a] = pd.to_numeric(df[a], errors="coerce")
    return df

from itertools import combinations
import pingouin as pg
import numpy as np

def welch_pairs(d, adj):
    rows = []
    conds = sorted(d["condition"].unique())
    for a, b in combinations(conds, 2):
        xa = d.loc[d["condition"] == a, adj].values
        xb = d.loc[d["condition"] == b, adj].values
        res = pg.ttest(xa, xb, correction=True)   
        g = pg.compute_effsize(xa, xb, eftype="hedges")  
        rows.append({
            "cond_A": a, "cond_B": b,
            "mean_A": xa.mean(), "mean_B": xb.mean(),
            "diff": xa.mean() - xb.mean(),
            "hedges_g": g,
            "p_raw": res["p_val"].iloc[0],   
        })
    return rows
    
def between_conditions(df):
    rows = []
    for block, conds in BLOCKS.items():
        sub_b = df[df["condition"].isin(conds)]
        for model in sorted(sub_b["model_id"].unique()):
            dm = sub_b[sub_b["model_id"] == model]
            for adj in ADJECTIVES:
                d = dm[["condition", adj]].dropna()
                counts = d.groupby("condition")[adj].agg(["count", "nunique"])
                valid = counts[(counts["count"] >= MIN_N) & (counts["nunique"] >= 2)].index
                d = d[d["condition"].isin(valid)]
                if d["condition"].nunique() < 2:
                    continue
                for r in welch_pairs(d, adj):
                    rows.append({
                        "block": block, "model_id": model, "adjective": adj,
                        "dimension": "warmth" if adj in WARMTH else "competence",
                        **r,
                    })
    out = pd.DataFrame(rows)
    if len(out):
        rej, pf, _, _ = multipletests(out["p_raw"], alpha=0.05, method="fdr_bh")
        out["p_fdr"] = pf; out["sig_fdr"] = rej
        out["relevant"] = out["sig_fdr"] & (out["hedges_g"].abs() >= 0.2)
    return out


def highlight_double_crossover(out):
    rel = out[out["relevant"]].copy()
    rel["pair"] = rel["cond_A"] + " vs " + rel["cond_B"]
    found = False
    for (model, block, pair), g in rel.groupby(["model_id", "block", "pair"]):
        comp = g[g["dimension"] == "competence"]
        warm = g[g["dimension"] == "warmth"]
        if len(comp) == 0 or len(warm) == 0:
            continue
        sign_comp = np.sign(comp["diff"].mean())
        sign_warm = np.sign(warm["diff"].mean())
        if sign_comp != 0 and sign_warm != 0 and sign_comp != sign_warm:
            winner_comp = pair.split(" vs ")[0] if sign_comp > 0 else pair.split(" vs ")[1]
            winner_warm = pair.split(" vs ")[0] if sign_warm > 0 else pair.split(" vs ")[1]
            print(f"\n  {model} {pair}:")
            print(f"     higher competence -> {winner_comp}")
            print(f"     higher warmth    -> {winner_warm} ")
            found = True
    if not found:
        print("\n  No double crossover.")

In [ ]:
df = load_data()
out = between_conditions(df)
if len(out):
    out.to_csv("rq1_between_conditions.csv", index=False)